# Moment measures from a convex potential

This notebook generates `fig:moment-measure-forward-map`.  For a convex potential $u$ on the line, the same function defines both the log-concave source law
$$
\rho_u(x)=Z_u^{-1}e^{-u(x)}
$$
and the monotone map $x\mapsto u'(x)$ whose push-forward is the moment measure
$$
\mathfrak M(u)=(u')_\sharp\rho_u.
$$
The figure compares the Gaussian quadratic case with an asymmetric strongly convex potential, where the source density is shifted but the pushed moment measure remains centered.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display
from scipy.integrate import cumulative_trapezoid

ROOT = Path.cwd()
if ROOT.name == "notebooks-figures":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "notebooks-figures"))

from figure_style import (
    BLUE,
    GRAY,
    LIGHT_GRAY,
    RED,
    VIOLET,
    box_axes,
    figure_dir,
    interp_color,
    remove_axes,
    save_pdf,
    setup_matplotlib,
)

setup_matplotlib()
FIG_NAME = "moment-measure-forward-map"
OUT = figure_dir(FIG_NAME)
THUMB_DIR = ROOT / "notebooks-figures" / "thumbnails"
THUMB_DIR.mkdir(parents=True, exist_ok=True)
PNG_OUT = THUMB_DIR / f"{FIG_NAME}.png"

## Exact one-dimensional change of variables

In one dimension, strict convexity makes $u'$ increasing, so the push-forward density is computed without sampling noise:
$$
    m_u(y)=\frac{\rho_u(x)}{u''(x)},\qquad y=u'(x).
$$
We use dense quadrature only to normalize $e^{-u}$ and to place transport marks at source quantiles.

In [ ]:
x_grid = np.linspace(-4.6, 4.6, 5001)

cases = [
    {
        "key": "quadratic",
        "row": 0,
        "tau": 0.0,
        "lam": 0.0,
        "panel_color": interp_color(0.16),
    },
    {
        "key": "asymmetric",
        "row": 1,
        "tau": 0.18,
        "lam": 0.92,
        "panel_color": interp_color(0.72),
    },
]


def potential(x, tau, lam):
    return 0.5 * x**2 + 0.25 * tau * x**4 + lam * x


def grad_potential(x, tau, lam):
    return x + tau * x**3 + lam


def hess_potential(x, tau, lam):
    return 1.0 + 3.0 * tau * x**2


def normalize_density(x, u):
    w = np.exp(-(u - u.min()))
    z = np.trapezoid(w, x)
    return w / z


def normalized_cdf(x, rho):
    cdf = np.concatenate([[0.0], cumulative_trapezoid(rho, x)])
    cdf = cdf / cdf[-1]
    cdf = np.maximum.accumulate(cdf)
    return cdf

for case in cases:
    tau, lam = case["tau"], case["lam"]
    u = potential(x_grid, tau, lam)
    rho = normalize_density(x_grid, u)
    y = grad_potential(x_grid, tau, lam)
    m = rho / hess_potential(x_grid, tau, lam)
    cdf = normalized_cdf(x_grid, rho)
    q_levels = np.linspace(0.06, 0.94, 13)
    x_marks = np.interp(q_levels, cdf, x_grid)
    y_marks = grad_potential(x_marks, tau, lam)
    mass = np.trapezoid(m, y)
    mean_y = np.trapezoid(y * m, y)

    case.update({
        "u": u,
        "rho": rho,
        "y": y,
        "m": m,
        "x_marks": x_marks,
        "y_marks": y_marks,
        "mass": mass,
        "mean_y": mean_y,
    })

[(c["key"], c["mass"], c["mean_y"]) for c in cases]

## Exported panels

The PDFs contain no embedded titles.  The LaTeX figure adds the row and column labels; the panels only show the densities, the rescaled potentials, and the induced monotone transport marks.

In [ ]:
def style_density_axis(ax, xlim=(-4.05, 4.05)):
    box_axes(ax)
    ax.set_xlim(*xlim)
    ax.set_xticks([-3, 0, 3])
    ax.set_yticks([])
    ax.tick_params(axis="x", labelsize=7, pad=1)
    ax.axhline(0, color="#333333", lw=0.55)


def plot_source_panel(case):
    x = x_grid
    rho = case["rho"]
    u = case["u"]
    u_scaled = 0.92 * rho.max() * (u - u.min()) / np.quantile(u - u.min(), 0.975)
    u_scaled = np.clip(u_scaled, 0, 1.10 * rho.max())

    fig, ax = plt.subplots(figsize=(2.28, 1.58))
    ax.fill_between(x, 0, rho, color=RED, alpha=0.18, linewidth=0)
    ax.plot(x, rho, color=RED, lw=1.45)
    ax.plot(x, u_scaled, color=GRAY, lw=0.98, alpha=0.78)
    ax.fill_between(x, 0, u_scaled, color=LIGHT_GRAY, alpha=0.10, linewidth=0)
    style_density_axis(ax)
    ax.set_ylim(-0.015 * rho.max(), 1.10 * max(rho.max(), u_scaled.max()))
    ax.text(0.02, 0.92, r"$\rho_u\propto e^{-u}$", transform=ax.transAxes, color=RED, fontsize=8, va="top")
    ax.text(0.98, 0.92, r"$u$", transform=ax.transAxes, color=GRAY, fontsize=8, va="top", ha="right")
    save_pdf(fig, OUT / f"{case['key']}-source.pdf")
    return fig


def plot_map_panel(case):
    x_marks = case["x_marks"]
    y_marks = case["y_marks"]
    fig, ax = plt.subplots(figsize=(2.28, 1.58))
    remove_axes(ax)
    xmin = min(x_marks.min(), y_marks.min()) - 0.45
    xmax = max(x_marks.max(), y_marks.max()) + 0.45
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(-0.17, 1.17)
    ax.plot([xmin, xmax], [0, 0], color=RED, lw=0.75, alpha=0.45)
    ax.plot([xmin, xmax], [1, 1], color=BLUE, lw=0.75, alpha=0.45)
    for k, (x0, y1) in enumerate(zip(x_marks, y_marks)):
        color = interp_color(k / max(len(x_marks) - 1, 1))
        ax.plot([x0, y1], [0, 1], color=VIOLET, alpha=0.50, lw=0.92)
        ax.scatter([x0], [0], s=11, color=RED, edgecolor="none", zorder=3)
        ax.scatter([y1], [1], s=11, color=BLUE, edgecolor="none", zorder=3)
    ax.text(xmin, -0.12, r"$x$", color=RED, fontsize=8, va="top")
    ax.text(xmin, 1.12, r"$y=u'(x)$", color=BLUE, fontsize=8, va="bottom")
    save_pdf(fig, OUT / f"{case['key']}-map.pdf")
    return fig


def plot_moment_panel(case):
    y = case["y"]
    m = case["m"]
    fig, ax = plt.subplots(figsize=(2.28, 1.58))
    ax.fill_between(y, 0, m, color=BLUE, alpha=0.18, linewidth=0)
    ax.plot(y, m, color=BLUE, lw=1.45)
    ax.axvline(0, color="#333333", lw=0.65, ls=(0, (2.0, 2.0)), alpha=0.85)
    style_density_axis(ax)
    ax.set_ylim(-0.015 * m.max(), 1.10 * m.max())
    ax.text(0.03, 0.92, r"$M(u)$", transform=ax.transAxes, color=BLUE, fontsize=8, va="top")
    ax.text(0.98, 0.92, r"$\int y\,dM(u)=0$", transform=ax.transAxes, color="#333333", fontsize=7.4, va="top", ha="right")
    save_pdf(fig, OUT / f"{case['key']}-moment.pdf")
    return fig

for case in cases:
    figs = [plot_source_panel(case), plot_map_panel(case), plot_moment_panel(case)]
    for fig in figs:
        plt.close(fig)

sorted(p.name for p in OUT.glob("*.pdf"))

## Thumbnail preview

The thumbnail below is only for the GitHub gallery.  The book uses the six individual PDF panels assembled directly in LaTeX.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(8.05, 3.60))
plt.subplots_adjust(wspace=0.22, hspace=0.28)

for row, case in enumerate(cases):
    # source/potential
    ax = axes[row, 0]
    rho = case["rho"]
    u = case["u"]
    u_scaled = 0.92 * rho.max() * (u - u.min()) / np.quantile(u - u.min(), 0.975)
    u_scaled = np.clip(u_scaled, 0, 1.10 * rho.max())
    ax.fill_between(x_grid, 0, rho, color=RED, alpha=0.18, linewidth=0)
    ax.plot(x_grid, rho, color=RED, lw=1.45)
    ax.plot(x_grid, u_scaled, color=GRAY, lw=0.98, alpha=0.78)
    style_density_axis(ax)
    ax.set_ylim(-0.015 * rho.max(), 1.10 * max(rho.max(), u_scaled.max()))

    # map
    ax = axes[row, 1]
    remove_axes(ax)
    x_marks, y_marks = case["x_marks"], case["y_marks"]
    xmin = min(x_marks.min(), y_marks.min()) - 0.45
    xmax = max(x_marks.max(), y_marks.max()) + 0.45
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(-0.17, 1.17)
    ax.plot([xmin, xmax], [0, 0], color=RED, lw=0.75, alpha=0.45)
    ax.plot([xmin, xmax], [1, 1], color=BLUE, lw=0.75, alpha=0.45)
    for k, (x0, y1) in enumerate(zip(x_marks, y_marks)):
        ax.plot([x0, y1], [0, 1], color=VIOLET, alpha=0.50, lw=0.92)
        ax.scatter([x0], [0], s=11, color=RED, edgecolor="none", zorder=3)
        ax.scatter([y1], [1], s=11, color=BLUE, edgecolor="none", zorder=3)

    # moment measure
    ax = axes[row, 2]
    ax.fill_between(case["y"], 0, case["m"], color=BLUE, alpha=0.18, linewidth=0)
    ax.plot(case["y"], case["m"], color=BLUE, lw=1.45)
    ax.axvline(0, color="#333333", lw=0.65, ls=(0, (2.0, 2.0)), alpha=0.85)
    style_density_axis(ax)
    ax.set_ylim(-0.015 * case["m"].max(), 1.10 * case["m"].max())

fig.savefig(PNG_OUT, dpi=220, bbox_inches="tight", pad_inches=0.035)
plt.close(fig)
display(Image(filename=str(PNG_OUT)))